##### Copyright 2026 Perceptron AI.

In [ ]:
# Licensed under the MIT License (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://opensource.org/licenses/MIT
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Get started with Isaac 0.1 — Image

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/perceptron-ai-inc/perceptron/blob/main/cookbook/quickstart/quickstart_isaac_0_1/quickstart_isaac_0_1.ipynb)

Ask Isaac 0.1 a question about an image and get a natural-language answer back. This quickstart configures the Perceptron SDK for the `isaac-0.1` model and runs a single Q&A call against a sample image URL — no local download needed.

Isaac 0.1 is the original open-weights image vision-language model from Perceptron, supported for existing integrations. Reasoning and Focus are **not** supported on 0.1 — see [Isaac 0.2](https://docs.perceptron.inc/isaac-0.2) or [Perceptron Mk1](https://docs.perceptron.inc/perceptron-mk1) for those capabilities.

![Studio scene sample](https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/capabilities/qna/studio_scene.webp)

**Note**

Set the `PERCEPTRON_API_KEY` environment variable (or edit the cell below) before running the call. You can swap the prompt or image (URL or local file path) for your own at any time.

----

## Install dependencies

In [ ]:
%pip install --upgrade perceptron --quiet

## Configure the Perceptron client
Authenticate once and point the SDK at Isaac 0.1.

In [ ]:
import os

from perceptron import configure, image, question

api_key = os.getenv("PERCEPTRON_API_KEY", "<your Perceptron API key>")
if not api_key or api_key.startswith("<"):
    raise RuntimeError("Set PERCEPTRON_API_KEY or replace the placeholder in this cell.")

configure(
    provider="perceptron",
    model="isaac-0.1",
    api_key=api_key,
)

IMAGE_URL = "https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/capabilities/qna/studio_scene.webp"

## Ask a question about the image

In [ ]:
QUESTION = "What is in this image?"

result = question(image(IMAGE_URL), QUESTION)
print(result.text)

## Ask for grounded boxes

Pass `expects="box"` to get bounding-box citations alongside the answer. Isaac returns each region in a normalized 0–1000 coordinate system; convert to pixels in your own code (see the [Image Q&A capability guide](https://docs.perceptron.inc/isaac-0.1/capabilities/image-qa) for a full PIL overlay walkthrough).

In [ ]:
GROUNDED = "Identify four features in the landscape."

result = question(image(IMAGE_URL), GROUNDED, expects="box")
print(result.text)

boxes = result.boxes or []
print(f"\n{len(boxes)} grounded region(s):")
for box in boxes:
    label = box.mention or "region"
    print(f"  - {label}: ({box.top_left.x:.0f}, {box.top_left.y:.0f}) → ({box.bottom_right.x:.0f}, {box.bottom_right.y:.0f})")

## Visualize the boxes

Download the image, overlay each returned box (converted from normalized 0–1000 coordinates into pixels), and display the result inline.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

from IPython.display import Image as IPyImage, display
from PIL import Image, ImageDraw

IMAGE_PATH = Path("studio_scene.webp")
if not IMAGE_PATH.exists():
    urlretrieve(IMAGE_URL, IMAGE_PATH)

img = Image.open(IMAGE_PATH).convert("RGB")
draw = ImageDraw.Draw(img)

for box in boxes:
    tl = (box.top_left.x / 1000 * img.width, box.top_left.y / 1000 * img.height)
    br = (box.bottom_right.x / 1000 * img.width, box.bottom_right.y / 1000 * img.height)
    draw.rectangle([tl, br], outline="crimson", width=3)
    label = box.mention or "region"
    draw.text((tl[0], max(tl[1] - 18, 0)), label, fill="crimson")

ANNOTATED = Path("studio_scene_annotated.png")
img.save(ANNOTATED)
display(IPyImage(filename=str(ANNOTATED)))

## Next steps
- Swap `IMAGE_URL` for your own image (URL or local file path — `image()` accepts both).
- Tune `QUESTION` toward inspections, retrieval, or scene description.
- Pass `expects="box"`, `"point"`, or `"polygon"` to receive grounded annotations alongside the answer.
- See the [Image Q&A capability guide](https://docs.perceptron.inc/isaac-0.1/capabilities/image-qa) for grounded responses with bounding-box overlays.
- For self-hosting, download the [isaac-0.1 weights](https://huggingface.co/PerceptronAI/Isaac-0.1) from Hugging Face.
- Ready to move beyond 0.1? Try the [Isaac 0.2 quickstart](https://github.com/perceptron-ai-inc/perceptron/blob/main/cookbook/quickstart/quickstart_isaac_0_2/quickstart_isaac_0_2.ipynb) for reasoning and Focus.